# AgentSystems-02 — Perception active : la valeur d'information

**Série AgentSystems** · Prolongement d'AgentSystems-01 (*perception → action*)

**Navigation** : [<< AgentSystems-01](AgentSystems-01-Perception-Vers-Action-Python.ipynb) | [Version Python](AgentSystems-02-Perception-Active-Valeur-Information-Python.ipynb) | [Index](README.md)
AgentSystems-01 a posé la boucle *percevoir → décider → agir*. Ce carnet ajoute la
couche qui fait de la perception un **choix** : l'agent ne lit pas tous les capteurs,
il observe **ce qui peut changer sa décision**. Chaque observation coûte ; la
**valeur d'information** d'une observation est la baisse du risque de décision qu'elle
rend possible. Ce carnet rend ce calcul exécutable : on mesure, capteur par capteur,
combien d'erreur il fait éviter, et on oppose la politique *active* (observer ce qui
compte) à une politique *naïve* (observer tout, dans l'ordre).

## Enjeu — de la perception passive à la perception active

Une perception **passive** lit tous les capteurs disponibles : elle est simple, mais
paie l'information qu'elle ne va pas utiliser. Une perception **active** raisonne
*avant* d'observer : « cette observation peut-elle changer ma décision ? ». Si oui, et
si la baisse d'erreur qu'elle promet vaut son coût, on l'achète ; sinon on s'abstient.

Le critère central est de **choisir une information parce qu'elle peut changer la
décision**. Son instrument est la **valeur d'information** (value of information) d'un
échantillon : l'écart entre le risque bayésien avant et après l'observation, moyenné sur
tous les résultats possibles du capteur.

## 0. La décision sous incertitude

Un état latent (`bonne` / `mauvaise` campagne), deux actions (`lancer` / `ne pas
lancer`), une matrice de perte. La seule inconnue qui gouverne la décision est
`q = P(mauvais)`.

In [1]:
// La decision sous incertitude : un etat latent, deux actions, une matrice de perte.
using System;
using System.Globalization;

const double L_ACT = 100.0;   // perte si on lance une campagne mauvaise
const double L_OPP = 30.0;    // perte d'opportunite si on ne lance pas une campagne bonne

double BayesRisk(double q)
{
    // Risque bayesien : perte attendue de la meilleure decision possible quand P(mauvais)=q.
    return Math.Min(q * L_ACT, (1.0 - q) * L_OPP);
}

string OptimalAction(double q)
{
    // Action bayesienne : lancer si le risque de lancer est le plus bas.
    return q * L_ACT < (1.0 - q) * L_OPP ? "lancer" : "ne pas lancer";
}

double q_seuil = L_OPP / (L_ACT + L_OPP); // on lance seulement si q est sous ce seuil
double q0 = 0.40;                         // a priori, la campagne a 40 % de chances d'etre mauvaise

Console.WriteLine($"Seuil bayesien q* = L_OPP/(L_ACT+L_OPP) = {q_seuil.ToString("F3", CultureInfo.InvariantCulture)}");
Console.WriteLine($"A priori q0 = {q0.ToString("F2", CultureInfo.InvariantCulture)}  ->  action optimale : {OptimalAction(q0)}");
Console.WriteLine($"Risque bayesien a priori : {BayesRisk(q0).ToString("F2", CultureInfo.InvariantCulture)}");

bool risqueConforme = BayesRisk(q0) == Math.Min(0.40 * L_ACT, 0.60 * L_OPP);
bool actionConforme = OptimalAction(q0) == "ne pas lancer";
Console.WriteLine($"Risque calculé = {BayesRisk(q0).ToString("F2", CultureInfo.InvariantCulture)} ; attendu = {Math.Min(0.40 * L_ACT, 0.60 * L_OPP).ToString("F2", CultureInfo.InvariantCulture)} : {(risqueConforme ? "CONFORME" : "ÉCART")}");
Console.WriteLine($"Action calculée = {OptimalAction(q0)} ; attendue = ne pas lancer : {(actionConforme ? "CONFORME" : "ÉCART")}");
if (!risqueConforme) throw new InvalidOperationException("Le risque bayésien diffère de la valeur attendue.");
if (!actionConforme) throw new InvalidOperationException("L'action optimale diffère de la valeur attendue.");

Seuil bayesien q* = L_OPP/(L_ACT+L_OPP) = 0.231


A priori q0 = 0.40  ->  action optimale : ne pas lancer


Risque bayesien a priori : 18.00


Risque calculé = 18.00 ; attendu = 18.00 : CONFORME


Action calculée = ne pas lancer ; attendue = ne pas lancer : CONFORME


### Lecture du résultat

Avec `q0 = 0.40`, au-dessus du seuil `0.231`, l'action bayésienne est **de ne pas
lancer** — et le risque bayésien (la perte attendue de la meilleure décision possible
avec l'information qu'on a) vaut **18**. Ce nombre est la référence : toute
observation qui le fait *baisser* porte de la valeur ; la question est de savoir de
combien, et au prix de quoi.

## 1. La valeur d'information d'un capteur

Un capteur est un modèle de bruit : `P(lecture | etat)`. Sa **valeur d'information**
est la baisse espérée du risque bayésien qu'il procure :

    EVOI(capteur) = RisqueBayesien(a priori) −  Σ_r  P(r) · RisqueBayesien(postérieur à r)

On ne la *cite* pas : on la **calcule** pour chaque capteur, en moyennant le risque
postérieur sur toutes les lectures que le capteur peut renvoyer. La valeur **nette**
soustrait le coût d'achat du capteur.

In [2]:
// Deux capteurs, meme cout d'achat, bruit different.
using System.Collections.Generic;
using System.Globalization;

// P(lecture "mauvaise" | etat) : le capteur "test de marche" est nettement plus informatif.
var CAPTEURS = new Dictionary<string, (double p_m_mauv, double p_m_bon, double cout)>
{
    ["test de marche"] = (0.90, 0.20, 10.0),
    ["analyse concurrentielle"] = (0.70, 0.45, 10.0)
};

(double pLectureMauvaise, double qMauvais, double qBon) PosteriorQ(double q, (double p_m_mauv, double p_m_bon, double cout) cap)
{
    // Posterieur P(mauvais | lecture) pour les deux lectures, a partir du prior q.
    double a = cap.p_m_mauv;
    double b = cap.p_m_bon;
    double p_l_m = a * q + b * (1.0 - q);             // P(lecture "mauvaise")
    double q_mauv = (a * q) / p_l_m;                  // P(mauvais | lecture "mauvaise")
    double q_bon = ((1.0 - a) * q) / (1.0 - p_l_m);  // P(mauvais | lecture "bonne")
    return (p_l_m, q_mauv, q_bon);
}

double Evoi(double q, (double p_m_mauv, double p_m_bon, double cout) cap)
{
    // Valeur d'information : baisse esperee du risque bayesien apportee par le capteur.
    var posterior = PosteriorQ(q, cap);
    double risque_apres = posterior.pLectureMauvaise * BayesRisk(posterior.qMauvais)
        + (1.0 - posterior.pLectureMauvaise) * BayesRisk(posterior.qBon);
    return BayesRisk(q) - risque_apres;
}

Console.WriteLine($"{"capteur",-24}{"EVOI",8}{"cout",7}{"valeur nette",14}");
foreach (var (nom, cap) in CAPTEURS)
{
    double v = Evoi(q0, cap);
    Console.WriteLine($"{nom,-24}{v.ToString("F2", CultureInfo.InvariantCulture),8}{cap.cout.ToString("F1", CultureInfo.InvariantCulture),7}{(v - cap.cout).ToString("F2", CultureInfo.InvariantCulture),14}");
}

double evoiMarche = Evoi(q0, CAPTEURS["test de marche"]);
double evoiConcurrentielle = Evoi(q0, CAPTEURS["analyse concurrentielle"]);
bool capteurConforme = evoiMarche > evoiConcurrentielle;
Console.WriteLine($"Comparaison EVOI : test de marche = {evoiMarche.ToString("F2", CultureInfo.InvariantCulture)} ; analyse concurrentielle = {evoiConcurrentielle.ToString("F2", CultureInfo.InvariantCulture)} : {(capteurConforme ? "CONFORME" : "ÉCART")}");
if (!capteurConforme) throw new InvalidOperationException("Le test de marche doit avoir une EVOI supérieure à celle de l'analyse concurrentielle.");

capteur                     EVOI   cout  valeur nette


test de marche             10.40   10.0          0.40


analyse concurrentielle     0.00   10.0        -10.00


Comparaison EVOI : test de marche = 10.40 ; analyse concurrentielle = 0.00 : CONFORME


### Lecture du résultat

Les deux capteurs coûtent le même prix, mais leur valeur n'est pas la même : le
**test de marché** baisse le risque de **10,40** points, tandis que l'**analyse
concurrentielle** en vaut **0** — son bruit est trop faible pour faire passer le
postérieur de l'autre côté du seuil de décision, elle ne change donc *aucune*
décision. C'est la leçon du tableau : **deux capteurs au même prix ne valent pas la
même information**, et un capteur peut en valoir *zéro*. La valeur *nette*
(EVOI − coût) décide de l'achat : le test de marché passe (+0,40), l'analyse
concurrentielle est un pur coût (−10,00).

## 2. La boucle de perception active

L'agent **actif** itère : (1) pour chaque capteur restant, calculer la valeur nette
au postérieur courant ; (2) acheter le capteur à la plus haute valeur nette *positive*
; (3) observer et mettre le postérieur à jour ; (4) répéter jusqu'à ce qu'aucun
capteur restant ait de valeur nette positive — alors on décide. Une politique
**naïve** achète, elle, tous les capteurs dans un ordre fixe, sans regarder leur
valeur. La trajectoire affichée est la réalisation la plus probable de chaque capteur
(la valeur d'information elle-même est déjà la moyenne sur toutes les lectures).

In [3]:
// def percevoir_actif(q, capteurs, budget_obs=5):
using System.Globalization;

(List<(string nom, string lecture)> sequence, double qFinal, string action, double coutTotal) PercevoirActif(
    double q,
    Dictionary<string, (double p_m_mauv, double p_m_bon, double cout)> capteurs,
    int budgetObs = 5)
{
    var sequence = new List<(string nom, string lecture)>();
    double cout = 0.0;
    var restants = new Dictionary<string, (double p_m_mauv, double p_m_bon, double cout)>(capteurs);

    for (int i = 0; i < budgetObs && restants.Count > 0; i++)
    {
        string meilleur = "";
        double meilleureValeurNette = double.NegativeInfinity;
        foreach (var (nom, cap) in restants)
        {
            double nette = Evoi(q, cap) - cap.cout;
            if (nette > meilleureValeurNette)
            {
                meilleur = nom;
                meilleureValeurNette = nette;
            }
        }

        if (meilleureValeurNette <= 0.0)
            break;

        var capteur = restants[meilleur];
        var posterior = PosteriorQ(q, capteur);
        string lecture = posterior.pLectureMauvaise >= 0.5 ? "mauvaise" : "bonne";
        q = lecture == "mauvaise" ? posterior.qMauvais : posterior.qBon;
        sequence.Add((meilleur, lecture));
        cout += capteur.cout;
        restants.Remove(meilleur);
    }

    return (sequence, q, OptimalAction(q), cout);
}

(List<(string nom, string lecture)> sequence, double qFinal, string action, double coutTotal) PercevoirNaif(
    double q,
    Dictionary<string, (double p_m_mauv, double p_m_bon, double cout)> capteurs,
    int budgetObs = 5)
{
    var sequence = new List<(string nom, string lecture)>();
    double cout = 0.0;
    foreach (var (nom, cap) in capteurs.Take(budgetObs))
    {
        var posterior = PosteriorQ(q, cap);
        string lecture = posterior.pLectureMauvaise >= 0.5 ? "mauvaise" : "bonne";
        q = lecture == "mauvaise" ? posterior.qMauvais : posterior.qBon;
        sequence.Add((nom, lecture));
        cout += cap.cout;
    }
    return (sequence, q, OptimalAction(q), cout);
}

var actif = PercevoirActif(q0, CAPTEURS);
var naif = PercevoirNaif(q0, CAPTEURS);
Console.WriteLine("POLITIQUE ACTIVE");
foreach (var (nom, lecture) in actif.sequence)
    Console.WriteLine($"  achete {nom}  (lecture: {lecture})");
Console.WriteLine($"  ->  decision: {actif.action}   |   cout total: {actif.coutTotal.ToString("F1", CultureInfo.InvariantCulture)}\n");
Console.WriteLine("POLITIQUE NAIVE");
foreach (var (nom, lecture) in naif.sequence)
    Console.WriteLine($"  achete {nom}  (lecture: {lecture})");
Console.WriteLine($"  ->  decision: {naif.action}   |   cout total: {naif.coutTotal.ToString("F1", CultureInfo.InvariantCulture)}");

bool coutConforme = actif.coutTotal < naif.coutTotal;
Console.WriteLine($"Coût actif = {actif.coutTotal.ToString("F1", CultureInfo.InvariantCulture)} ; coût naïf = {naif.coutTotal.ToString("F1", CultureInfo.InvariantCulture)} : {(coutConforme ? "CONFORME" : "ÉCART")}");
if (!coutConforme) throw new InvalidOperationException("La politique active doit coûter strictement moins cher que la politique naïve.");

POLITIQUE ACTIVE


  achete test de marche  (lecture: bonne)


  ->  decision: lancer   |   cout total: 10.0



POLITIQUE NAIVE


  achete test de marche  (lecture: bonne)


  achete analyse concurrentielle  (lecture: bonne)


  ->  decision: lancer   |   cout total: 20.0


Coût actif = 10.0 ; coût naïf = 20.0 : CONFORME


### Le résultat — observer ce qui compte

La politique **active** s'arrête après le seul capteur qui vaille son prix ; la
politique **naïve** achète les deux. Même décision finale, **coût plus bas** pour
l'active : elle a refusé de payer l'information que le second capteur n'apportait
plus. Une fois le premier lu, la boucle active s'arrête (sortie ci-dessus) : plus aucune lecture n'a une valeur nette positive. Ce n'est pas
l'information en soi qui compte, mais l'information qui *change la décision*.


## Synthèse — la perception comme choix

| | Perception passive (naïve) | Perception active (valeur d'information) |
|---|---|---|
| Critère d'achat | tous les capteurs, à l'aveugle | seulement si `EVOI > coût` |
| Coût | somme de tous les prix | somme des prix *qui valaient le coup* |
| Décision | après tout | dès que plus rien ne vaut son prix |
| Lien avec l'Epic | « percevoir » | « **choisir** une information parce qu'elle peut changer la décision » |

La boucle d'AgentSystems-01 (percevoir → décider → agir) devient une boucle où la
perception est **argumentée** : chaque capteur justifie son prix par la baisse
d'erreur qu'il rend possible.

## Exercices

In [4]:
// Exercice 1 — Le troisieme capteur
// On ajoute un capteur "enquete client" : P(l="mauvaise"|mauv)=0.95,
// P(l="mauvaise"|bon)=0.05, cout 25. Calculer sa valeur d'information et sa valeur
// nette a l'a priori q0, et dire si la politique active l'acheterait (avant le
// test de marche, apres, ou jamais).
(double? evoi, double? valeurNette, string decisionAchat) EnqueteClient()
{
    // TODO etudiant
    return (null, null, null);
}


In [5]:
// Exercice 2 — Le seuil de decision
// A quel a priori q0 la decision bascule-t-elle de "ne pas lancer" a "lancer" ?
// Exprimer q* en fonction de L_ACT et L_OPP, puis le verifier par le code
// (le seuil doit satisfaire optimal_action(q* - e) = "lancer" et
// optimal_action(q* + e) = "ne pas lancer" pour un petit e).
double? SeuilDecision()
{
    // TODO etudiant
    return null;
}


In [6]:
// Exercice 3 - La valeur d information marginale
// Comparer la valeur d information de l analyse concurrentielle calculee a priori
// et apres la lecture bonne du premier capteur : toutes deux valent 0 dans cet
// exemple (posterior 1/13). Expliquer pourquoi elles sont nulles ici, puis relier
// a la condition d arret de la boucle active (aucune valeur nette positive).
string ValeurMarginale()
{
    // TODO etudiant
    return null;
}


## À retenir

- La **valeur d'information** d'une observation est la baisse du risque bayésien qu'elle
  rend possible, moyennée sur ses résultats — on la *calcule*, on ne la cite pas.
- Une observation a une **valeur nette** positive seulement si `EVOI > coût` ; la
  perception active n'achète que celles-là.
- La boucle active s'arrête quand **aucune valeur nette n'est positive** ; dans cet
  exemple, la valeur d'information de l'analyse concurrentielle est nulle avant comme
  après la lecture — l'arrêt vient de l'absence de gain, pas d'une chute mesurée.
- C'est le critère de l'Epic : *choisir une information parce qu'elle peut changer la
  décision* — la perception devient un argument, pas un réflexe.

## Bibliographie

- X. Chen, A. Guestrin, *Bayesian Decision Theory: A Perspective on the Uncertainty of
  Machine Learning*, 2019 — le risque bayésien comme objet de décision.
- I. Russo, *Entropy Inference: Common Information Measures for Continuous Variables*,
  2016 — la quantification de l'information.
- Carnet frère : **AgentSystems-01 — Perception vers Action** (la boucle percevoir →
  décider → agir posée en amont de ce carnet).

> Carnet issu de l'EPIC #26930.